In [1]:
!pip show pandas 

Name: pandas
Version: 2.2.3
Summary: Powerful data structures for data analysis, time series, and statistics
Home-page: https://pandas.pydata.org
Author: 
Author-email: The Pandas Development Team <pandas-dev@python.org>
License: BSD 3-Clause License

 Copyright (c) 2008-2011, AQR Capital Management, LLC, Lambda Foundry, Inc. and PyData Development Team
 All rights reserved.

 Copyright (c) 2011-2023, Open source contributors.

 Redistribution and use in source and binary forms, with or without
 modification, are permitted provided that the following conditions are met:

 * Redistributions of source code must retain the above copyright notice, this
   list of conditions and the following disclaimer.

 * Redistributions in binary form must reproduce the above copyright notice,
   this list of conditions and the following disclaimer in the documentation
   and/or other materials provided with the distribution.

 * Neither the name of the copyright holder nor the names of its
   contribut

In [2]:
import sys
!{sys.executable} -m pip install pandas openpyxl

In [9]:
import gurobipy as gp
from gurobipy import GRB
import pandas as pd
import math

In [10]:
#parameters
timelimit = 600

In [11]:
#load data
nodes_df = pd.read_excel("customers.xlsx",sheet_name="Nodes")
request_df = pd.read_excel("customers.xlsx",sheet_name="Requests")
fleet_df = pd.read_excel("customers.xlsx",sheet_name="Fleet")  

In [12]:
#definitions
#depot info
depot_id = int(nodes_df[nodes_df['type'] == 0]['id'].iloc[0])

#list customer IDs
customer_ids = request_df["id"].tolist()

#coordinates for computing euclidean distance
coordinates = nodes_df.set_index("id")[["cx","cy"]].to_dict("index")

#adding time and capacities
demand = {} #quantity requested at node
service = {} #service time (installation time)
earliest = {} #earliest allowed start
latest = {} #latest allowed start

for _, row in request_df.iterrows():
    node = int(row['id'])
    demand[node] = float(row['quantity'])
    service[node] = float(row['service_time'])
    earliest[node] = float(row['start'])
    latest[node] = float(row['end'])

#depot gets 0 demand and trivial time window
demand[depot_id] = 0
service[depot_id] = 0
earliest[depot_id] = 0
latest[depot_id] = 9999999


In [13]:
# Euclidean distance 
def dist(i, j):
    x = coordinates[i]["cx"] - coordinates[j]["cx"]
    y = coordinates[i]["cy"] - coordinates[j]["cy"]
    distance = int(round(math.sqrt(x**2 + y**2)))
    return distance

In [16]:
#travel time
travel_time = {}
for i in range(M_big):
    for j in range(M_big):
        travel_time[i,j] = dist(node_ids[i], node_ids[j])
    

In [17]:
#add capacity
capacity = float(fleet_df['capacity'].iloc[0])

In [18]:
#instance loop
timelimit = 600
instance_sizes = [5,10]

for n in instance_sizes:
    print("\n==============================")
    print("Building instance with", n, "customers")
    print("==============================")
    
    node_ids = [depot_id] + customer_ids[:n]
    M_big = len(node_ids)
    print("Selected node_ids:", node_ids)
    
    MTZE_model = gp.Model('VRPTW_MTZ')
    MTZE_model.setParam('TimeLimit', timelimit)

#decision variables
x =  MTZE_model.addVars(M_big, M_big, vtype=GRB.BINARY, name="x")
load =  MTZE_model.addVars(M_big, vtype=GRB.CONTINUOUS, name="L")
T =  MTZE_model.addVars(M_big, vtype=GRB.CONTINUOUS, name="T")

#objective
MTZE_model.setObjective(
    gp.quicksum(dist(node_ids[i], node_ids[j]) * x[i,j]
                for i in range(M_big) for j in range(M_big)),
    GRB.MINIMIZE
)

print('Created model for', n, 'customers')
print('Selected node_ids', node_ids)


Building instance with 5 customers
Selected node_ids: [0, 1, 2, 3, 4, 5]
Set parameter TimeLimit to value 600

Building instance with 10 customers
Selected node_ids: [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10]
Set parameter TimeLimit to value 600
Created model for 10 customers
Selected node_ids [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10]


In [36]:
#constraints MTZ - extended capacity and time

# MTZ constraints needs to be relaxed - currently forces one big TSP tour when we want different tours now

#for customers we want it to go in and out once, but depot degree can be relaxed
    # MTZ_model.addConstr(u[0] == 0)  REMOVED

#customer only constraint
for i in range(1, M_big):
    MTZE_model.addConstr(gp.quicksum(x[i,j] for j in range(M_big) if j != i) == 1 )
    MTZE_model.addConstr(gp.quicksum(x[j,i] for j in range(M_big) if j != i) == 1 )
# #depot flow balance??
MTZE_model.addConstr(
    gp.quicksum(x[0,j] for j in range(1, M_big)) ==
    gp.quicksum(x[j,0] for j in range(1, M_big))
)

#capacity constraints
MTZE_model.addConstr(load[0] == 0 )
for i in range(1, M_big):
    MTZE_model.addConstr(load[i] >= demand[node_ids[i]])
    MTZE_model.addConstr(load[i] <= capacity)

for i in range(1, M_big):
    for j in range(1, M_big):
        if i != j:
            MTZE_model.addConstr(
                load[i] + demand[node_ids[j]] - load[j] <= capacity * (1 - x[i,j])
)
#time windows
MTZE_model.addConstr(T[0] == 0)

for i in range(1, M_big):
    MTZE_model.addConstr(T[i] >= earliest[node_ids[i]])
    MTZE_model.addConstr(T[i] <= latest[node_ids[i]])

In [47]:
#consistency in travel times needed INFEASIBLE
BIGM = 100000
for i in range(M_big):
    for j in range(M_big):
        if i != j:
            MTZE_model.addConstr(
                T[i] + service[node_ids[i]] + travel_time[i,j] 
                <= T[j] + BIGM * (1 - x[i,j]))

In [42]:
#print('customer_ids =', customer_ids)
#print('node_ids =', node_ids)
#print('service keys =', service.keys())

In [48]:
#solve the model
MTZE_model.optimize()
print("Status code:", MTZE_model.status)

if MTZE_model.status in [GRB.OPTIMAL, GRB.TIME_LIMIT]:
    print("Objective (total distance):", MTZE_model.objVal)

#nr of vehicles used
    used_vehicles = sum(
        1 for j in range(1, M_big) if x[0,j].X > 0.5
        )
    print("# vehicles used:", used_vehicles)
else:
    print("No fesible solution - skipping .X access.")
    print("Number of constraints:", MTZE_model.numConstrs)

Gurobi Optimizer version 12.0.3 build v12.0.3rc0 (win64 - Windows 11.0 (26100.2))

CPU model: Intel(R) Core(TM) i7-9750H CPU @ 2.60GHz, instruction set [SSE2|AVX|AVX2]
Thread count: 6 physical cores, 12 logical processors, using up to 12 threads

Non-default parameters:
TimeLimit  600

Optimize a model with 876 rows, 143 columns and 3113 nonzeros
Model fingerprint: 0x22ac66eb
Variable types: 22 continuous, 121 integer (121 binary)
Coefficient statistics:
  Matrix range     [1e+00, 1e+05]
  Objective range  [1e+00, 2e+01]
  Bounds range     [1e+00, 1e+00]
  RHS range        [1e+00, 1e+05]

MIP start from previous solve did not produce a new incumbent solution

Presolve removed 62 rows and 70 columns
Presolve time: 0.00s

Explored 0 nodes (0 simplex iterations) in 0.02 seconds (0.00 work units)
Thread count was 1 (of 12 available processors)

Solution count 0

Model is infeasible
Best objective -, best bound -, gap -
Status code: 3
No fesible solution - skipping .X access.
Number of cons

In [44]:
#printing
print("Objective =", MTZE_model.objVal)
print("Number of constraints:", MTZE_model.numConstrs)

AttributeError: Unable to retrieve attribute 'objVal'